In [1]:
from pathlib import Path
import json
from dataclasses import asdict
from itertools import product

from rabl.machine_learning.tuner import GridSearchConfig, TrialResult, count_grid_combinations, run_grid_search, test_best_model
from rabl.machine_learning.lstm_pipeline import (
    build_datasets,
    train_with_fallback,
    cleanup_cuda,
    save_forecast_profiles_pdf,
)

In [2]:
lookback_datasets = {
    # N = 2001 datasets
    #12: Path("../outputs/datasets/scaled_split/lstm_merged_batches_0001-0002_k12_minmax_train0.70_val0.15_test0.15.h5"),
    #8:  Path("../outputs/datasets/scaled_split/lstm_merged_batches_0001-0002_k8_minmax_train0.70_val0.15_test0.15.h5"),
    #4:  Path("../outputs/datasets/scaled_split/lstm_merged_batches_0001-0002_k4_minmax_train0.70_val0.15_test0.15.h5"),
    # N = 501 datasets
    12: Path("../outputs/datasets/scaled_split/lstm_merged_batches_0003_k12_minmax_train0.70_val0.15_test0.15.h5"),
}

for lb, p in lookback_datasets.items():
    print(f"lookback={lb} exists={p.exists()} path={p}")

lookback=12 exists=True path=../outputs/datasets/scaled_split/lstm_merged_batches_0003_k12_minmax_train0.70_val0.15_test0.15.h5


In [3]:
out_dir = Path("../outputs/ml_results/hp_tuning/N501/tuning_k12_part2/")
out_dir.mkdir(parents=True, exist_ok=True)

config = GridSearchConfig(
    lookback_datasets=lookback_datasets,
    learning_rates=[2e-4, 3e-4, 5e-4],
    batch_sizes=[256, 512, 1024],
    n_lstm_values=[1],
    hidden_lstm_values=[32, 64, 128],
    hidden_fc_values=[32, 128, 256, 384],
    n_fc=1,
    epochs=50,
    seed=123,
    out_dir=out_dir,
    prefer_gpu=True,
    verbose=1,
    lstm_dropout=0.3,
    preload_train_to_device=True,
    early_stopping_patience=10,
    early_stopping_min_delta=1e-7,
    restore_best_weights=True,
    step_lr_step_size=30,
    step_lr_gamma=0.5,
)

print("out_dir:", config.out_dir)
print("Total trials:", count_grid_combinations(config))

out_dir: ../outputs/ml_results/hp_tuning/N501/tuning_k12_part2
Total trials: 108


In [ ]:
results, best = run_grid_search(config)
print("Best trial:", best)

Total combinations: 108
[1/108] lookback=12, lr=0.0002, batch_size=256, n_lstm=1, hidden_lstm=32, hidden_fc=32
Using GPU: NVIDIA A100-SXM4-40GB

Deterministic seed set to: 123
LSTMRegressor architecture:
  Input features: 14
  LSTM layers (1): hidden_size=32, dropout=0.3
  FC layers (1): [32]
  Output targets: 13



/home/burnloga2/miniforge3/envs/rabl/lib/python3.11/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "
Preloading train batches: 1370batch [00:06, 214.77batch/s]


Preloaded 1370 training batches to cuda:0 in 6.38s.


Train 1/50: 100%|██████████| 1370/1370 [00:03<00:00, 386.24batch/s, loss=4.06614e-04]
Val 1/50: 294batch [00:01, 181.01batch/s]


Epoch 1/50 - loss: 2.05567e-02 - val_loss: 1.95881e-03 - lr: 2.000e-04 - data_wait: 0.01s - h2d: 0.00s - compute: 3.10s - val_time: 1.63s - epoch_total: 4.74s - preloaded: True - preload_time: 6.38s - max_cuda_mem: 351.85 MB - io_h5: 2.76s - io_cast: 0.03s - io_profiles: 700 - io_samples: 350700 - io_tput: 125798.93 samp/s - io_frac_of_data_wait: 22576.93%
Epoch 1/50 total_time_s: 4.74


...

Train 27/50: 100%|██████████| 1370/1370 [00:03<00:00, 439.13batch/s, loss=1.43164e-06]
Val 27/50: 294batch [00:01, 216.96batch/s]


Epoch 27/50 - loss: 3.70807e-06 - val_loss: 1.15411e-05 - lr: 3.000e-04 - data_wait: 0.01s - h2d: 0.00s - compute: 2.74s - val_time: 1.36s - epoch_total: 4.11s - preloaded: True - preload_time: 5.31s - max_cuda_mem: 351.85 MB - io_h5: 0.00s - io_cast: 0.00s - io_profiles: 0 - io_samples: 0 - io_tput: 0.00 samp/s - io_frac_of_data_wait: 0.00%
Epoch 27/50 total_time_s: 4.11


Train 28/50: 100%|██████████| 1370/1370 [00:03<00:00, 416.29batch/s, loss=1.10489e-06]
Val 28/50: 294batch [00:01, 226.71batch/s]


Epoch 28/50 - loss: 3.41576e-06 - val_loss: 1.11933e-05 - lr: 3.000e-04 - data_wait: 0.01s - h2d: 0.00s - compute: 2.81s - val_time: 1.30s - epoch_total: 4.12s - preloaded: True - preload_time: 5.31s - max_cuda_mem: 351.85 MB - io_h5: 0.00s - io_cast: 0.00s - io_profiles: 0 - io_samples: 0 - io_tput: 0.00 samp/s - io_frac_of_data_wait: 0.00%
Epoch 28/50 total_time_s: 4.12


Train 29/50:  82%|████████▏ | 1119/1370 [00:02<00:00, 460.08batch/s, loss=3.08357e-06]

In [8]:
test_metrics = test_best_model(config, best)
print("Test timing metrics:", test_metrics)

/home/burnloga2/github/RABL/src/rabl/machine_learning/tuner.py:246: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location="cpu")


LSTMRegressor architecture:
  Input features: 14
  LSTM layers (1): hidden_size=64, dropout=0.3
  FC layers (1): [64]
  Output targets: 13



Forecast profiles: 150profile [00:30,  4.89profile/s, mae_avg=8.437753e+02]



Testing timing summary: total_fetch: 0.0001s -  total_test: 28.3744s -  avg_fetch_profile: 0.0000s -  avg_inference_profile: 0.1892s -  save_time: 0.1450s

Saved best-model test outputs to: ../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0028_lb12_lr0.0003_bs64_nl1_hl64_hf64/best_model_test
Test timing metrics: {'avg_fetch_profile_s': 4.085828550159931e-07, 'avg_inference_profile_s': 0.18916197590685138, 'total_fetch_s': 6.128742825239897e-05, 'total_test_s': 28.37435767345596, 'save_time_s': 0.14498310000635684}


In [4]:
save_forecast_profiles_pdf(
    forecast_h5_path=Path("../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128/best_model_test/rolling_forecasts.h5"),
    output_pdf_path=Path("../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128/best_model_test/rolling_forecasts.pdf"),
)

Plotted 10/150
Plotted 20/150
Plotted 30/150
Plotted 40/150
Plotted 50/150
Plotted 60/150
Plotted 70/150
Plotted 80/150
Plotted 90/150
Plotted 100/150
Plotted 110/150
Plotted 120/150
Plotted 130/150
Plotted 140/150
Plotted 150/150
Saved forecast PDF to: ../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128/best_model_test/rolling_forecasts.pdf


# Test best model from any trials (not just the best trial)

In [1]:
from pathlib import Path
from rabl.machine_learning.tuner import GridSearchConfig, test_model_from_trial_dir

config = GridSearchConfig(
    # Not used by trial lookup itself, but required by dataclass:
    lookback_datasets={},
    learning_rates=[],
    batch_sizes=[],
    n_lstm_values=[],
    hidden_lstm_values=[],
    hidden_fc_values=[],

    # Must point to the same tuning output root that contains grid_search_results.json
    out_dir=Path("../outputs/ml_results/hp_tuning/N501/tuning_k12/"),

    # Used when rebuilding/testing the selected model:
    seed=123,
    n_fc=1,
    lstm_dropout=0.0,
    test_output_dirname="best_model_test",
    use_tqdm=True,
)

metrics = test_model_from_trial_dir(
    config,
    Path("../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128/")
)

print(metrics)

Testing trial #38: /home/burnloga2/github/RABL/outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128 (lookback=12)


/home/burnloga2/github/RABL/src/rabl/machine_learning/tuner.py:248: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location="cpu")


LSTMRegressor architecture:
  Input features: 14
  LSTM layers (1): hidden_size=64, dropout=0.0
  FC layers (1): [128]
  Output targets: 13



Forecast profiles: 150profile [00:31,  4.71profile/s, mae_avg=8.638102e+02]



Testing timing summary: total_fetch: 0.0001s -  total_test: 29.5760s -  avg_fetch_profile: 0.0000s -  avg_inference_profile: 0.1972s -  save_time: 0.1241s

Saved best-model test outputs to: ../outputs/ml_results/hp_tuning/N501/tuning_k12/trial_0038_lb12_lr0.0003_bs128_nl1_hl64_hf128/best_model_test
{'avg_fetch_profile_s': 4.560624559720357e-07, 'avg_inference_profile_s': 0.1971726070328926, 'total_fetch_s': 6.840936839580536e-05, 'total_test_s': 29.575959464302287, 'save_time_s': 0.12410164508037269}
